# Dialogue Summarization Project - Phase 1

This notebook explores the **SAMSum dialogue summarization dataset**.

In Phase 1, we will:

- Load the dataset from Hugging Face
- Understand the train, validation, and test splits
- Check missing values
- Analyze dialogue and summary lengths
- Find common words
- Try simple text cleaning and tokenization
- Save plots in the `results/` folder

This notebook is written with beginner-friendly comments so each step is easy to follow.

## 1. Install Required Libraries

If you are using Google Colab, run the next cell once. If the libraries are already installed, you can skip it.

In [ ]:
# For Google Colab, uncomment and run this line if needed.
# The exclamation mark runs a terminal command inside the notebook.
# !pip install datasets pandas numpy matplotlib seaborn nltk transformers torch

## 2. Import Libraries

We import Python libraries for data loading, data handling, text processing, and visualization.

In [1]:
# Standard Python libraries
import os
import sys
from collections import Counter

# Data analysis libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Hugging Face library used to download datasets
from datasets import load_dataset

# Add the src folder to Python path so we can import our own files.
# This works when the notebook is run from the project root folder.
sys.path.append("src")

# Import our custom helper functions.
from preprocess import clean_text, tokenize_text, count_words, preprocess_example
from utils import create_results_folder, get_most_common_words, save_histogram

# Make plots look a little nicer.
sns.set_theme(style="whitegrid")

# Create the results folder if it does not exist.
results_path = create_results_folder("results")

print("Libraries imported successfully!")
print(f"Plots will be saved in: {results_path.resolve()}")

Libraries imported successfully!
Plots will be saved in: C:\Users\Bristi\OneDrive\Documents\dialogue summarizer\dialogue_summarization_project\results


## 3. Load the SAMSum Dataset

The SAMSum dataset contains messenger-style dialogues and short human-written summaries.

We load it directly from Hugging Face using the required dataset name.

In [2]:
# Load the SAMSum dataset from Hugging Face.
# This may take a little time the first time because it downloads the data.
dataset = load_dataset("knkarthick/samsum")

# Print the dataset object to see available splits and columns.
dataset

README.md: 0.00B [00:00, ?B/s]

C:\Users\Bristi\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Bristi\.cache\huggingface\hub\datasets--knkarthick--samsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

## 4. Dataset Overview

A Hugging Face dataset usually has multiple splits. For this project, we expect:

- `train`
- `validation`
- `test`

Let us count how many samples are present in each split.

In [3]:
# Count samples in each dataset split.
split_counts = []

for split_name in dataset.keys():
    split_counts.append({
        "split": split_name,
        "number_of_samples": len(dataset[split_name])
    })

# Convert the result to a DataFrame for a neat table.
split_counts_df = pd.DataFrame(split_counts)
split_counts_df

,split,number_of_samples
0,train,14731
1,validation,818
2,test,819


In [4]:
# Convert each split into a pandas DataFrame.
# DataFrames are easy to inspect and analyze.
train_df = dataset["train"].to_pandas()
validation_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

# Show the first 5 training examples.
train_df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJ...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i...,"Sam is confused, because he overheard Rick com..."


## 5. Understand Columns

Before doing analysis, it is useful to inspect column names and data types.

In [5]:
# Display column names.
print("Columns in train data:")
print(train_df.columns.tolist())

# Display data types for each column.
print("\nData types:")
print(train_df.dtypes)

Columns in train data:
['id', 'dialogue', 'summary']

Data types:
id          object
dialogue    object
summary     object
dtype: object


## 6. View Sample Dialogue and Summary

Let us print one full example so we understand what the input and output look like.

In [6]:
# Select one example from the training data.
sample = train_df.iloc[0]

print("DIALOGUE:")
print(sample["dialogue"])

print("\nSUMMARY:")
print(sample["summary"])

DIALOGUE:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

SUMMARY:
Amanda baked cookies and will bring Jerry some tomorrow.


## 7. Missing Value Analysis

Missing values can cause errors during preprocessing and model training.

We check if any column contains empty or missing values.

In [7]:
# Check missing values in all splits.
for split_name, df in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    print(f"Missing values in {split_name} split:")
    print(df.isnull().sum())
    print("-" * 40)

Missing values in train split:
id          0
dialogue    0
summary     0
dtype: int64
----------------------------------------
Missing values in validation split:
id          0
dialogue    0
summary     0
dtype: int64
----------------------------------------
Missing values in test split:
id          0
dialogue    0
summary     0
dtype: int64
----------------------------------------


## 8. Add Length Features

To understand the dataset better, we create new columns:

- `dialogue_word_count`: number of words in each dialogue
- `summary_word_count`: number of words in each summary
- `dialogue_char_count`: number of characters in each dialogue
- `summary_char_count`: number of characters in each summary

In [ ]:
# Make copies so we do not accidentally change the original DataFrames.
train_df = train_df.copy()
validation_df = validation_df.copy()
test_df = test_df.copy()

# Create word count columns using our helper function.
train_df["dialogue_word_count"] = train_df["dialogue"].apply(count_words)
train_df["summary_word_count"] = train_df["summary"].apply(count_words)

# Create character count columns using simple string length.
train_df["dialogue_char_count"] = train_df["dialogue"].apply(lambda text: len(str(text)))
train_df["summary_char_count"] = train_df["summary"].apply(lambda text: len(str(text)))

# Show the new columns.
train_df[["dialogue_word_count", "summary_word_count", "dialogue_char_count", "summary_char_count"]].head()

## 9. Basic Length Statistics

The `describe()` function gives useful statistics such as mean, minimum, maximum, and percentiles.

In [ ]:
# Show statistics for dialogue and summary word counts.
train_df[["dialogue_word_count", "summary_word_count"]].describe()

## 10. Dialogue Length Distribution

This histogram shows how long the dialogues usually are.

The plot is also saved in the `results/` folder.

In [ ]:
save_histogram(
    dataframe=train_df,
    column="dialogue_word_count",
    title="Dialogue Length Distribution",
    xlabel="Number of Words in Dialogue",
    output_path="results/dialogue_length_distribution.png",
    bins=50,
)

## 11. Summary Length Distribution

This histogram shows how long the summaries usually are.

In [ ]:
save_histogram(
    dataframe=train_df,
    column="summary_word_count",
    title="Summary Length Distribution",
    xlabel="Number of Words in Summary",
    output_path="results/summary_length_distribution.png",
    bins=40,
)

## 12. Compare Dialogue and Summary Lengths

A scatter plot helps us see whether longer dialogues usually have longer summaries.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=train_df,
    x="dialogue_word_count",
    y="summary_word_count",
    alpha=0.4
)
plt.title("Dialogue Length vs Summary Length")
plt.xlabel("Dialogue Word Count")
plt.ylabel("Summary Word Count")
plt.tight_layout()
plt.savefig("results/dialogue_vs_summary_length.png")
plt.show()

## 13. Most Common Words in Dialogues

Common words help us understand what appears frequently in conversations.

This simple analysis does not remove stopwords, so words like `i`, `you`, and `the` may appear often.

In [ ]:
# Find the 20 most common words in training dialogues.
common_dialogue_words = get_most_common_words(train_df["dialogue"], top_n=20)

# Convert to a DataFrame for easier plotting.
common_dialogue_df = pd.DataFrame(common_dialogue_words, columns=["word", "count"])
common_dialogue_df

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=common_dialogue_df, x="count", y="word")
plt.title("Most Common Words in Dialogues")
plt.xlabel("Count")
plt.ylabel("Word")
plt.tight_layout()
plt.savefig("results/most_common_dialogue_words.png")
plt.show()

## 14. Most Common Words in Summaries

Now we repeat the same analysis for summaries.

In [ ]:
common_summary_words = get_most_common_words(train_df["summary"], top_n=20)
common_summary_df = pd.DataFrame(common_summary_words, columns=["word", "count"])
common_summary_df

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=common_summary_df, x="count", y="word")
plt.title("Most Common Words in Summaries")
plt.xlabel("Count")
plt.ylabel("Word")
plt.tight_layout()
plt.savefig("results/most_common_summary_words.png")
plt.show()

## 15. Text Cleaning Examples

Text cleaning makes raw text more consistent.

Our simple cleaning function:

- Converts text to lowercase
- Removes extra spaces
- Replaces line breaks with spaces
- Removes website links

In [ ]:
# Take one dialogue example.
raw_dialogue = train_df.iloc[0]["dialogue"]
cleaned_dialogue = clean_text(raw_dialogue)

print("Original dialogue:")
print(raw_dialogue)

print("\nCleaned dialogue:")
print(cleaned_dialogue)

## 16. Tokenization Examples

Tokenization means splitting text into smaller units.

Here we use a very simple word tokenizer. Later, when using transformer models, we will use model-specific tokenizers.

In [ ]:
# Tokenize one dialogue and one summary.
dialogue_tokens = tokenize_text(train_df.iloc[0]["dialogue"])
summary_tokens = tokenize_text(train_df.iloc[0]["summary"])

print("First 30 dialogue tokens:")
print(dialogue_tokens[:30])

print("\nSummary tokens:")
print(summary_tokens)

## 17. Preprocess One Full Example

The `preprocess_example()` function shows how we can package several preprocessing steps together.

In [ ]:
processed_sample = preprocess_example(
    dialogue=train_df.iloc[0]["dialogue"],
    summary=train_df.iloc[0]["summary"]
)

processed_sample

## 18. Save a Small Cleaned Preview

It is useful to save a few cleaned examples so we can inspect them later.

In [ ]:
# Create a small preview with original and cleaned text.
preview_df = train_df[["id", "dialogue", "summary"]].head(10).copy()
preview_df["clean_dialogue"] = preview_df["dialogue"].apply(clean_text)
preview_df["clean_summary"] = preview_df["summary"].apply(clean_text)

# Save the preview CSV file in the results folder.
preview_df.to_csv("results/cleaned_preview.csv", index=False)

preview_df.head()

## 19. Speaker Distribution Analysis

SAMSum dialogues usually write each turn as `Speaker: message`.

We can count the unique speaker names in each dialogue and then compare 2-person dialogues with multi-person dialogues.

In [ ]:
import re

def extract_speakers(dialogue):
    """Extract unique speaker names from dialogue lines formatted as 'Speaker: message'."""
    speakers = []

    for line in str(dialogue).split("\n"):
        match = re.match(r"^\s*([^:]{1,50})\s*:", line)
        if match:
            speaker = match.group(1).strip()
            speakers.append(speaker)

    return sorted(set(speakers))


def classify_speaker_count(number_of_speakers):
    """Group dialogues into 2-person, multi-person, or other categories."""
    if number_of_speakers == 2:
        return "2-person"
    if number_of_speakers > 2:
        return "multi-person"
    return "other"


# Add speaker features to the training data.
train_df["speakers"] = train_df["dialogue"].apply(extract_speakers)
train_df["number_of_speakers"] = train_df["speakers"].apply(len)
train_df["speaker_group"] = train_df["number_of_speakers"].apply(classify_speaker_count)

# Display examples of the extracted speaker information.
train_df[["id", "speakers", "number_of_speakers", "speaker_group"]].head()

In [ ]:
# Count how many dialogues belong to each speaker group.
speaker_distribution_df = (
    train_df["speaker_group"]
    .value_counts()
    .rename_axis("speaker_group")
    .reset_index(name="number_of_dialogues")
)

# Add percentages for easier interpretation.
speaker_distribution_df["percentage"] = (
    speaker_distribution_df["number_of_dialogues"] / len(train_df) * 100
).round(2)

# Save the table for the report.
speaker_distribution_df.to_csv("results/speaker_distribution.csv", index=False)

speaker_distribution_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=speaker_distribution_df,
    x="speaker_group",
    y="number_of_dialogues"
)
plt.title("Speaker Distribution in Training Dialogues")
plt.xlabel("Dialogue Type")
plt.ylabel("Number of Dialogues")
plt.tight_layout()
plt.savefig("results/speaker_distribution.png")
plt.show()

## 20. Phase 1 Summary

In this notebook, we completed Phase 1 of the project:

- Loaded the SAMSum dataset
- Checked dataset splits and sample counts
- Inspected sample dialogues and summaries
- Checked missing values
- Analyzed dialogue and summary lengths
- Counted 2-person and multi-person dialogue distribution
- Created histograms and bar plots
- Practiced text cleaning and tokenization
- Saved plots, speaker distribution, and a cleaned preview in the `results/` folder

The next phase can focus on preparing data for model training.